# NB12

Rebuild manuscript tables from pipeline outputs.

In [ ]:
# Rebuild manuscript tables

import os, glob, time
from pathlib import Path
import numpy as np
import pandas as pd

from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.section import WD_ORIENT
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

BASE = Path(os.environ.get("MES_BASE_DIR", "."))
REV=(BASE/"Manuscript data"/"Tables"/"Revision")
if not REV.exists(): REV=(BASE/"Manuscript Data"/"Tables"/"Revision")
OUT=REV.parent/"Final"; OUT.mkdir(parents=True, exist_ok=True)

def find(*names):
    for n in names:
        hits=list(REV.glob(n))
        if hits: return hits[0]
    return None
def rd(path, sheet=0):
    return pd.read_excel(path, sheet_name=sheet)

# collected tables: list of dicts {num,title,caption,df,footnote}
TABLES=[]
def register(num,title,df,caption,footnote=""):
    if df is None or len(df)==0:
        print(f"  [{num}] EMPTY, skipped"); return
    # round floats for display, but keep p/q values in scientific precision
    df=df.copy()
    def _is_pcol(name):
        n=str(name).lower()
        return any(k in n for k in ["p_","_p","pval","p(","q_","q(","_q","mw_p","slope_p","spearman_p"]) or n in ("p","q")
    for c in df.columns:
        if df[c].dtype.kind in "fc":
            if _is_pcol(c):
                df[c]=df[c].apply(lambda v: v if pd.isna(v) else float(f"{v:.3g}"))
            else:
                df[c]=df[c].apply(lambda v: round(v,4) if pd.notna(v) else v)
    tag=("Table_"+str(num)).replace(' ','_').replace('/','_')
    csv=OUT/f"{tag}_{title.replace(' ','_').replace('/','_').replace(',','')[:40]}.csv"
    df.to_csv(csv, index=False)
    TABLES.append({"num":str(num),"title":title,"caption":caption,"df":df,"footnote":footnote})
    print(f"  [Table {num}] {title}: {df.shape[0]}x{df.shape[1]} -> {csv.name}")

def banner(t): print("\n"+"="*70+f"\n{t}\n"+"="*70)

print(f"Reading pipeline outputs from: {REV}")
print(f"Writing CSVs + Word file to:   {OUT}")

# T1 module annotation (FDR-correct axes) from A4
banner("T1 module annotation")
f=find("Supp_Rev_A4_AxisPriors_Expanded.xlsx")
if f:
    try: best=rd(f,"best_axis_expanded")
    except Exception: best=rd(f,0)
    def col(df,*c):
        for x in c:
            if x in df.columns: return x
        low={k.lower():k for k in df.columns}
        for x in c:
            if x.lower() in low: return low[x.lower()]
        return None
    qcol=col(best,"pval_BH_expanded","pval_BH","q","p_BH","BH","q_BH")
    acol=col(best,"best_axis_expanded","axis","best_axis","Best Enrichment Axis","best_enrichment_axis")
    fcol=col(best,"fold_expanded","fold","fold_enrichment")
    if qcol and acol:
        qv=pd.to_numeric(best[qcol],errors="coerce")
        best["FDR_significant"]=qv<0.05
        best["axis_final"]=np.where(qv<0.05, best[acol].astype(str), "Mixed / unassigned (q>=0.05)")
        # reorder so the corrected label sits next to the evidence
        front=[c for c in [acol,fcol,qcol,"FDR_significant","axis_final"] if c]
        mcol=col(best,"module","MES")
        ordered=([mcol] if mcol else [])+front+[c for c in best.columns if c not in front and c!=mcol]
        best=best[ordered]
    register("1","Consensus Microenvironmental Education Signature (MES) Modules",best,
             "Table 1. Consensus Microenvironmental Education Signature (MES) Modules. "
             "Axis assignment reflects formal enrichment testing with Benjamini-Hochberg correction; "
             "modules not reaching q < 0.05 for any axis are labeled mixed/unassigned.",
             "Fold enrichment and BH-corrected p values from hypergeometric axis-enrichment testing.")
else:
    print("  MISSING Supp_Rev_A4_AxisPriors_Expanded.xlsx")

# T2 MES-tolerance coupling per cohort (uniform negative) from A10
banner("T2 MES-tolerance coupling")
fA10=find("Supp_Rev_A10_Microglia_Purity.xlsx")
if fA10:
    strat=rd(fA10,"MES_vs_Tol_by_Purity")
    g=strat.groupby(["dataset","MES"])["r"].mean().reset_index()
    piv=g.pivot(index="MES",columns="dataset",values="r").reset_index()
    register("S5","MES-Tolerance Correlations Across Four Microglia Cohorts",piv,
             "Table S5. MES-Tolerance Correlations Across Four Microglia Cohorts. Donor-level Spearman correlations between each MES module and the "
             "tolerance-positioning score across four microglia cohorts. All associations are negative, "
             "indicating higher MES expression tracks a less homeostatic microglial state.",
             "Values are mean adjusted Spearman rho across purity strata; see Table 3 for the purity gradient.")
else:
    print("  MISSING Supp_Rev_A10_Microglia_Purity.xlsx")

# T3 purity-stratified Q1 vs Q4 from A10
banner("T3 purity-stratified coupling")
if fA10:
    strat=rd(fA10,"MES_vs_Tol_by_Purity")
    rows=[]
    for (ds,mes),sub in strat.groupby(["dataset","MES"]):
        q1=sub[sub["purity_quartile"].astype(str).str.contains("Q1")]["r"]
        q4=sub[sub["purity_quartile"].astype(str).str.contains("Q4")]["r"]
        if len(q1) and len(q4):
            rows.append({"dataset":ds,"MES":mes,"r_Q1_low":float(q1.values[0]),
                         "r_Q4_high":float(q4.values[0]),
                         "stronger_with_purity":bool(float(q4.values[0])<float(q1.values[0]))})
    register("S20","MES-Tolerance Coupling Stratified by Microglial Purity",pd.DataFrame(rows),
             "Table S20. MES-Tolerance Coupling Stratified by Microglial Purity. Coupling in the lowest (Q1) versus highest (Q4) microglial purity "
             "quartiles per cohort. A more negative Q4 value indicates the association strengthens with "
             "microglial purity.",
             "Purity defined by microglia marker-score quartile within each cohort.")

# T4 tissue specificity from S2
banner("T4 tissue specificity")
f=find("Supp_Rev_S2_Thymus_vs_Controls.xlsx")
if f:
    register("S4","Tissue Specificity of MES-Tolerance Coupling (Thymus versus Six Control Tissues)",rd(f,0),
             "Table S4. Tissue Specificity of MES-Tolerance Coupling: Thymus versus Six Control Tissues. Mean absolute MES-tolerance "
             "coupling for thymus-derived modules versus modules derived by an identical procedure from "
             "six other human tissues, with fold-enrichment and one-sided Mann-Whitney p values.",
             "Modules from each control tissue built with matched K and HVG settings; immune-compartment cells only.")
else:
    print("  MISSING Supp_Rev_S2_Thymus_vs_Controls.xlsx")

# T5 GR null - per-module interaction from canonical panel (NB11) + external cohort (NB12)
banner("T5 GR moderation null (per-module interaction)")
gr_parts=[]
fN=find("Supp_Rev_N2_Canonical_GR_Interaction.xlsx")
if fN:
    d=rd(fN,0).copy()
    # merge in power/equivalence columns from N3 (separate sheet in NB11)
    fN3=find("Supp_Rev_N3_Power_Equivalence.xlsx")
    if fN3:
        n3=rd(fN3,0)
        keepn3=[c for c in ["dataset","MES","adequately_powered","equivalent_to_null","MDE_80pct","TOST_p"] if c in n3.columns]
        if "dataset" in d.columns and "MES" in d.columns and len(keepn3)>2:
            d=d.merge(n3[keepn3], on=["dataset","MES"], how="left", suffixes=("","_n3"))
            # prefer N3 values where the interaction table left them blank
            for c in ["adequately_powered","equivalent_to_null"]:
                if c+"_n3" in d.columns:
                    d[c]=d[c].where(d[c].notna(), d[c+"_n3"])
                    d=d.drop(columns=[c+"_n3"])
    d.insert(0,"source","canonical panel (NB11)")
    gr_parts.append(d)
else:
    print("  MISSING Supp_Rev_N2_Canonical_GR_Interaction.xlsx")
fP=find("Supp_Rev_P2_External_GR_Interaction.xlsx")
if fP:
    d=rd(fP,0)
    d=d.copy(); d.insert(0,"source","external GSE174367 (NB12)")
    gr_parts.append(d)
else:
    print("  MISSING Supp_Rev_P2_External_GR_Interaction.xlsx")

if gr_parts:
    grdf=pd.concat(gr_parts, ignore_index=True, sort=False)
    # keep the columns that matter for a null-moderation table, if present
    want=["source","dataset","MES","beta_int","se_int","ci_lo","ci_hi","p_int","q_BH",
          "above_small_effect","adequately_powered","equivalent_to_null"]
    keep=[c for c in want if c in grdf.columns]
    grdf=grdf[keep] if keep else grdf
    # drop rows with no interaction estimate (cohorts lacking donor structure: MS, Tuddenham)
    if "beta_int" in grdf.columns:
        before=len(grdf)
        grdf=grdf[pd.to_numeric(grdf["beta_int"],errors="coerce").notna()].reset_index(drop=True)
        print(f"  T5: dropped {before-len(grdf)} rows with no interaction estimate (insufficient-donor cohorts)")
    register("S21","Glucocorticoid Receptor Moderation: Per-Module Interaction Tests",grdf,
             "Table S21. Glucocorticoid Receptor Moderation, Per-Module Interaction Tests. GR moderation of the MES-tolerance association, tested as a "
             "per-module interaction (tolerance ~ MES + GR + MES x GR + covariates) with a canonical 20-gene "
             "GR panel and replicated in an external cohort (GSE174367). All interaction coefficients are "
             "small (standardized |beta| < 0.04, below a pre-specified 0.10 effect-size bound). Although some "
             "coefficients reach statistical significance in the largest cohort owing to very large cell "
             "numbers, all are statistically equivalent to a null effect by two one-sided tests, and the "
             "analysis is adequately powered to detect a small effect had one been present. We therefore "
             "interpret GR moderation as absent.",
             "Standardized MES x GR interaction coefficient with 95% CI; q by Benjamini-Hochberg across "
             "modules within cohort; equivalence assessed by TOST against a +/-0.10 standardized bound. "
             "above_small_effect = |beta| exceeds 0.10; equivalent_to_null = statistically within +/-0.10.")
else:
    print("  T5 has no source files")

# T6 Visium from A12
banner("T6 Visium spatial")
f=find("Supp_Rev_A12_Visium_MicrogliaWeighted.xlsx")
if f:
    try: v=rd(f,"long")
    except Exception: v=rd(f,0)
    keep=[c for c in v.columns if c in ("MES","r_unweighted","r_weighted_pearson","r_top30pct_microglia")]
    if "MES" in keep:
        g=v.groupby("MES")[ [c for c in keep if c!="MES"] ].mean().reset_index()
    else:
        g=v
    register("S6","Spatial Validation: GSE220442 Visium",g,
             "Table S6. Spatial Validation, GSE220442 Visium. Spot-level MES-tolerance correlations in Visium spatial transcriptomics, including "
             "microglia-weighted estimates, confirming the inverse association holds in intact tissue.",
             "Microglia-weighted correlations upweight spots with higher inferred microglial content.")
else:
    print("  MISSING Supp_Rev_A12_Visium_MicrogliaWeighted.xlsx")

# T7 heterogeneity from A15
banner("T7 heterogeneity")
f=find("Supp_Rev_A15_MS_drop_meta.xlsx")
if f:
    try: h=rd(f,"MS_drop_sensitivity")
    except Exception: h=rd(f,0)
    h=h.copy()
    # round I2/pooled columns to 1-2 dp for readability
    for c in h.columns:
        lc=str(c).lower()
        if h[c].dtype.kind in "fc":
            if "i2" in lc or "i²" in lc or "delta_i" in lc:
                h[c]=h[c].round(1)
            else:
                h[c]=h[c].round(4)
    # append a SUMMARY row: median + range of the two I2 columns so the honest
    # central value is visible inside the table, not just inferable.
    i2w=[c for c in h.columns if "i2" in str(c).lower() and ("with" in str(c).lower())]
    i2n=[c for c in h.columns if "i2" in str(c).lower() and ("no" in str(c).lower())]
    if i2w and i2n:
        import numpy as _np
        wv=pd.to_numeric(h[i2w[0]],errors="coerce").dropna()
        nv=pd.to_numeric(h[i2n[0]],errors="coerce").dropna()
        summ={col:"" for col in h.columns}
        mcol=[c for c in h.columns if str(c).lower() in ("mes","module")]
        if mcol: summ[mcol[0]]="SUMMARY (median; range)"
        summ[i2w[0]]=f"{_np.median(wv):.1f} ; {wv.min():.0f}-{wv.max():.0f}"
        summ[i2n[0]]=f"{_np.median(nv):.1f} ; {nv.min():.0f}-{nv.max():.0f}"
        h=pd.concat([h, pd.DataFrame([summ])], ignore_index=True)
    register("S10","Meta-Analysis of MES Heterogeneity (DerSimonian-Laird Random Effects)",h,
             "Table S10. Meta-Analysis of MES Heterogeneity, DerSimonian-Laird Random Effects. Random-effects meta-analysis heterogeneity (I-squared) for the MES-tolerance "
             "association by module, with and without the single-donor multiple sclerosis cohort. "
             "Heterogeneity was substantial for most modules (median I-squared approximately 87 percent "
             "with the MS cohort and 79 percent without; full per-module range shown).",
             "I-squared and pooled rho by DerSimonian-Laird random-effects meta-analysis; the single-donor "
             "MS cohort is assessed as a sensitivity exclusion. Summary row reports median and range across modules.")
else:
    print("  MISSING Supp_Rev_A15_MS_drop_meta.xlsx")

# T8 stage coupling from NB13 Q2
banner("T8 stage coupling")
f=find("Supp_Rev_Q2_Coupling_vs_Stage_Trend.xlsx")
if f:
    s=rd(f,0)
    sigcol=[c for c in s.columns if "sig" in c.lower()]
    show=s[s[sigcol[0]]==True] if sigcol and s[sigcol[0]].any() else s
    register("S22","MES-Tolerance Coupling versus Neuropathological Stage",show,
             "Table S22. MES-Tolerance Coupling versus Neuropathological Stage. Association between MES-tolerance coupling strength and neuropathological stage. A "
             "significant negative trend is present in SEA-AD (CERAD) but does not replicate across cohorts; "
             "reported as a single-cohort trend.",
             "Per-donor coupling regressed on ordinal stage, weighted by sqrt(cells); BH-corrected within cohort.")
else:
    print("  MISSING Supp_Rev_Q2_Coupling_vs_Stage_Trend.xlsx")

# ASSEMBLE THE WORD FILE (journal-ready)
banner("Assembling Word file")

def _set_cell_border(cell, **kwargs):
    """Set individual cell borders. kwargs like top={'sz':8,'val':'single'}."""
    tcPr=cell._tc.get_or_add_tcPr()
    tcB=tcPr.find(qn('w:tcBorders'))
    if tcB is None:
        tcB=OxmlElement('w:tcBorders'); tcPr.append(tcB)
    for edge in ('top','bottom','left','right'):
        if edge in kwargs:
            spec=kwargs[edge]
            el=tcB.find(qn('w:'+edge))
            if el is None:
                el=OxmlElement('w:'+edge); tcB.append(el)
            for k,v in spec.items():
                el.set(qn('w:'+k), str(v))

def _no_borders(cell):
    """Remove all borders from a cell (set to nil)."""
    for edge in ('top','bottom','left','right','insideH','insideV'):
        _set_cell_border(cell, **{edge:{'val':'nil'}})

def set_cell_margins(cell, top=30, bottom=30, left=80, right=80):
    tcPr=cell._tc.get_or_add_tcPr()
    m=OxmlElement('w:tcMar')
    for tag,val in [('w:top',top),('w:bottom',bottom),('w:start',left),('w:end',right)]:
        e=OxmlElement(tag); e.set(qn('w:w'),str(val)); e.set(qn('w:type'),'dxa'); m.append(e)
    tcPr.append(m)

def set_col_widths(table, widths_emu):
    table.autofit=False; table.allow_autofit=False
    for row in table.rows:
        for i,w in enumerate(widths_emu):
            if i < len(row.cells): row.cells[i].width=w

RULE_TOP={'sz':12,'val':'single','color':'000000','space':0}     # ~1.5pt top rule
RULE_MID={'sz':6,'val':'single','color':'000000','space':0}      # ~0.75pt under header
RULE_BOT={'sz':12,'val':'single','color':'000000','space':0}     # ~1.5pt bottom rule

FONT="Times New Roman"; FSZ=10; FSZ_FOOT=8.5

def _style_run(r, size=FSZ, bold=False, italic=False):
    r.font.name=FONT; r.font.size=Pt(size); r.bold=bold; r.italic=italic
    # ensure east-asian + ascii both map to TNR
    rPr=r._element.get_or_add_rPr()
    rFonts=rPr.find(qn('w:rFonts'))
    if rFonts is None:
        rFonts=OxmlElement('w:rFonts'); rPr.append(rFonts)
    for a in ('w:ascii','w:hAnsi','w:cs'):
        rFonts.set(qn(a), FONT)

def add_table(doc, t, content_width_dxa):
    df=t["df"]; ncol=len(df.columns); nrow=len(df)
    # caption
    cap=doc.add_paragraph()
    cap.paragraph_format.space_before=Pt(10); cap.paragraph_format.space_after=Pt(3)
    _style_run(cap.add_run(t["caption"]), size=FSZ)
    # table (no built-in style => no default grid; we draw only 3 rules)
    tbl=doc.add_table(rows=1, cols=ncol)
    tbl.alignment=WD_TABLE_ALIGNMENT.CENTER
    # widths
    per=int(content_width_dxa/ncol)
    first=int(per*1.5) if ncol>3 else per
    rest=int((content_width_dxa-first)/(ncol-1)) if ncol>1 else content_width_dxa
    widths=[first]+[rest]*(ncol-1) if ncol>1 else [content_width_dxa]
    widths_emu=[w*635 for w in widths]
    # header row
    hdr=tbl.rows[0].cells
    for i,c in enumerate(df.columns):
        hdr[i].text=""
        p=hdr[i].paragraphs[0]; p.paragraph_format.space_before=Pt(1); p.paragraph_format.space_after=Pt(1)
        _style_run(p.add_run(str(c)), size=FSZ, bold=True)
        _no_borders(hdr[i]); set_cell_margins(hdr[i])
        _set_cell_border(hdr[i], top=RULE_TOP, bottom=RULE_MID)   # top rule + header rule
    # body rows
    for ridx,(_,row) in enumerate(df.iterrows()):
        cells=tbl.add_row().cells
        is_last=(ridx==nrow-1)
        for i,c in enumerate(df.columns):
            val=row[c]; cn=str(c).lower()
            is_p=any(k in cn for k in ["p_","_p","pval","p(","q_","q(","_q","mw_p","slope_p","spearman_p"]) or cn in ("p","q")
            if pd.isna(val): txt=""
            elif isinstance(val,float):
                if is_p: txt=f"{val:.2e}" if (val!=0 and abs(val)<1e-3) else f"{val:.4g}"
                else: txt=f"{val:g}"
            else: txt=f"{val}"
            cells[i].text=""
            p=cells[i].paragraphs[0]; p.paragraph_format.space_before=Pt(1); p.paragraph_format.space_after=Pt(1)
            _style_run(p.add_run(txt), size=FSZ)
            _no_borders(cells[i]); set_cell_margins(cells[i])
            if is_last:
                _set_cell_border(cells[i], bottom=RULE_BOT)   # bottom rule on last row
    set_col_widths(tbl, widths_emu)
    # footnote
    if t["footnote"]:
        fn=doc.add_paragraph()
        fn.paragraph_format.space_before=Pt(2); fn.paragraph_format.space_after=Pt(8)
        _style_run(fn.add_run(t["footnote"]), size=FSZ_FOOT, italic=True)

doc=Document()
# set Normal style to TNR 10 as a baseline
try:
    normal=doc.styles["Normal"]; normal.font.name=FONT; normal.font.size=Pt(FSZ)
except Exception: pass
sec=doc.sections[0]
sec.orientation=WD_ORIENT.LANDSCAPE
sec.page_width=Inches(11); sec.page_height=Inches(8.5)
sec.left_margin=sec.right_margin=Inches(0.7); sec.top_margin=sec.bottom_margin=Inches(0.7)
content_width_dxa=int((11-1.4)*1440)

title=doc.add_paragraph()
_style_run(title.add_run("Compiled Manuscript Tables (pipeline-verified)"), size=13, bold=True)
sub=doc.add_paragraph()
_style_run(sub.add_run("Tables named for their manuscript slot. Three-line format, Times New Roman. "
                       "Verify, then paste each into its corresponding location."), size=9, italic=True)

def _order_key(t):
    n=str(t["num"])
    if n=="1": return (0,0)
    if n.startswith("S"):
        try: return (1,int(n[1:]))
        except: return (2,0)
    return (3,0)
for t in sorted(TABLES, key=_order_key):
    add_table(doc, t, content_width_dxa)

docpath=OUT/"Manuscript_Tables_Compiled.docx"
doc.save(docpath)
print(f"\n  Word file: {docpath}")
print(f"  Tables included: {[t['num'] for t in sorted(TABLES,key=lambda x:x['num'])]}")
banner("DONE")
print(f"CSVs and Manuscript_Tables_Compiled.docx are in: {OUT}")
print("Send me the .docx and I will verify every value before you paste.")
